# Matrix Alignment



## Purpose

The purpose of this notebook is to determine whether every participant listed in the cohort metadata corresponds to exactly one FCM file.

The audit checks participant identifiers and FCM filenames without inspecting the contents of the matrices.

The required correspondence is:

**225 metadata participants ↔ 225 unique FCM files**

The audit will identify missing, additional, duplicated, malformed, or incorrectly named FCM files.

In [1]:
from pathlib import Path

import pandas as pd

from lemon_connectivity.alignment import align_participants_to_fcm

# Load Data 

In [2]:
fcm_data = Path(
    "../data/external/Curvature-FCN-Aging/DATA/FCM"
)

cohort_path = Path(
    "../data/external/Curvature-FCN-Aging/DATA/cohort_information.tsv"
)

cohort_metadata = pd.read_csv(
    cohort_path,
    sep="\t",
    dtype={"sub_id": "string"},
)

alignment_result = align_participants_to_fcm(
    cohort_metadata,
    fcm_data,
)
fcm_inventory = alignment_result.inventory

## FCM directory inventory

The reusable alignment pipeline inventories the total number of directory
entries, regular files, subdirectories, extensions, unexpected file types, and
system files. Only non-system `.txt` files are treated as candidate FCM files.

Only directory names and filenames are inspected at this stage.

In [3]:
alignment_result.directory_report

total_entries                                   225
regular_files                                   225
non_system_txt_files                            225
files_with_expected_prefix_and_txt_extension    225
unexpected_file_extensions                       []
unexpected_subdirectories                        []
system_entries                                   []
dtype: object

### Interpretation

The FCM directory inventory identified `225 regular files` and no unexpected subdirectories, file extensions, or system files. All 225 files use the expected `.txt` extension.

## FCM filename validation

The FCM directory was inspected at the filename level to determine whether the available matrices follow the expected naming convention.

The expected filename format is:

`fcm_sub_<subject-ID>.txt`

For example:

`fcm_sub_32301.txt`

The validation checks whether each filename:

* starts with the required `fcm_sub_` prefix;
* ends with the `.txt` extension;
* contains a participant ID between the prefix and extension;
* contains only numeric characters in the participant ID;
* can be parsed unambiguously into the original participant ID.




In [4]:
alignment_result.filename_report

non_system_txt_files_checked                 225
filenames_matching_expected_pattern          225
valid_fcm_filenames                          225
malformed_or_unparseable_filenames             0
matrix_ids_normalized                        225
pattern_matches_with_invalid_canonical_id      0
dtype: object

### Interpretation

The FCM filenames were checked against the expected `fcm_sub_<subject-ID>.txt`. Valid filenames were parsed to recover the original participant IDs, while any malformed filenames were recorded separately for investigation.

##  Data Integrity and Normalization

In [5]:
alignment_result.filename_report.loc[
    [
        "matrix_ids_normalized",
        "pattern_matches_with_invalid_canonical_id",
    ]
]

matrix_ids_normalized                        225
pattern_matches_with_invalid_canonical_id      0
dtype: object

## interpretation
Participant identifiers are treated as `strings` 

The metadata participant IDs are loaded as `strings`, and participant IDs parsed from FCM filenames are also retained as `strings`. No participants are renumbered, and gaps in the original identifier sequence are preserved.

The original participant identifier supplied by Yadav remains unchanged. A canonical identifier is created separately by zero-padding the original ID to six digits and adding the sub- prefix.

The canonical conversion is checked for collisions to ensure that distinct participant IDs do not resolve to the same canonical identifier.


# Check uniqueness of matrix IDs.

In [6]:
alignment_result.uniqueness_report


valid_fcm_files              225
parsed_matrix_ids            225
unique_parsed_matrix_ids     225
duplicate_filename_ids         0
canonical_collision_count      0
duplicate_canonical_ids        0
dtype: object

## interpretation

The participant IDs extracted from the validated FCM filenames were checked for uniqueness and one-to-one representation.

The audit compares the total number of FCM files with the number of parsed and unique participant IDs. It also identifies participant IDs appearing in more than one filename, exact duplicate filenames, and any collisions introduced by canonical ID conversion.


## Cross-Referencing and Set Validation

In [7]:
alignment_result.set_report


metadata_rows                     225
metadata_ids_not_canonicalized      0
unique_metadata_ids               225
unique_matrix_ids                 225
metadata_without_matrices           0
matrices_without_metadata           0
matched_participants              225
dtype: object

## interpretation

The participant IDs from the cohort metadata were compared with the participant IDs parsed from the FCM filenames.

The comparison was performed in both directions to identify participants present in the metadata without a corresponding FCM file and FCM files whose participant IDs are absent from the metadata.


# Confirm one-to-one cardinality

In [8]:
alignment_result.cardinality_report


metadata_row_count_equals_one_for_all    True
matrix_file_count_equals_one_for_all     True
perfect_one_to_one_matches                225
cardinality_violations                      0
dtype: object

## interpretation

The participant-level mapping was checked to confirm that each participant occurs exactly once in the cohort metadata and exactly once among the FCM files.

The audit verifies that every metadata participant has one corresponding matrix file, that every matrix participant has one metadata record, and that the participant identifiers represented in both sources agree. A row receives the
alignment status `matched` only when all one-to-one conditions are satisfied.

## The participant-level alignment

In [9]:
alignment_report = alignment_result.alignment_table
alignment_summary = alignment_result.alignment_summary

print("=== AGGREGATE ALIGNMENT RESULTS ===")
print(alignment_summary)

output_dir = Path("../data/interim")
output_dir.mkdir(parents=True, exist_ok=True)
alignment_report.to_csv(
    output_dir / "participant_alignment_table.csv",
    index=False,
)

=== AGGREGATE ALIGNMENT RESULTS ===
matched_participants                  225
metadata_without_matrices               0
matrices_without_metadata               0
duplicate_metadata_ids                  0
duplicate_matrix_ids                    0
duplicate_metadata_and_matrix_ids       0
invalid_filenames                       0
invalid_matrix_ids                      0
invalid_metadata_ids                    0
missing_metadata_ids                    0
anomalies                               0
total_discrepancies                     0
alignment_passed                     True
dtype: object


## interpretation

The reusable pipeline created a participant-level table containing one row for
each canonical ID found in either source, plus explicit rows for invalid or
missing identifiers. It records source presence, row/file counts, the matrix
filename and path, filename validity, and final alignment status.

The complete table is retained locally because it contains participant
identifiers. It is not intended for inclusion in the repository.



## Discrepancies

In [10]:
discrepancies = alignment_report.loc[
    alignment_report["alignment_status"].ne("matched")
]

print("=== DISCREPANCY AUDIT REPORT ===")
if discrepancies.empty:
    matched = int(alignment_summary["matched_participants"])
    print(
        "Success: Zero discrepancies found. "
        f"All {matched} participants are perfectly matched!"
    )
else:
    print(f"Warning: Found {len(discrepancies)} alignment issue(s):")
    print(
        discrepancies[
            ["canonical_id", "matrix_filename", "alignment_status"]
        ]
    )


=== DISCREPANCY AUDIT REPORT ===
Success: Zero discrepancies found. All 225 participants are perfectly matched!


### Interpretation

The discrepancy audit identified no records with an alignment status other than `matched`. This indicates that every participant represented in the alignment table has a corresponding FCM file and that no metadata-only, matrix-only, duplicate-matrix, or invalid-filename cases were detected.

The participant-level metadata-to-FCM filename alignment therefore passed the discrepancy check.
